# LeetCode 1475 — Final Prices With a Special Discount in a Shop bằng Python

**Link đề:** https://leetcode.com/problems/final-prices-with-a-special-discount-in-a-shop/
**Độ khó:** Easy · **Chủ đề:** Array, Stack, Monotonic Stack (ngăn xếp đơn điệu)

```python
class Solution:
    def finalPrices(self, prices: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Đọc đúng đề: với mỗi món `i`, tìm **chỉ số `j` nhỏ nhất** sao cho `j > i` và `prices[j] <= prices[i]`.
2. Viết lời giải trực giác (brute force) bằng hai vòng lặp lồng nhau + `break`.
3. Hiểu cấu trúc **stack** (ngăn xếp) trong Python bằng `list.append` / `list.pop` / `stack[-1]`.
4. Nắm kỹ thuật **monotonic stack** để giải bài "phần tử nhỏ hơn-hoặc-bằng kế tiếp" (next smaller or equal element) trong `O(n)`.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, tự giải các biến thể "next greater / previous smaller".

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — giá cuối cùng sau khi được giảm

Cho mảng số nguyên `prices`, `prices[i]` là giá của món thứ `i` trong cửa hàng.

Cửa hàng có khuyến mãi đặc biệt: nếu mua món `i`, bạn được giảm một khoản bằng `prices[j]`, trong đó `j` là **chỉ số nhỏ nhất** thỏa mãn:

- `j > i` (món nằm **sau** món `i`), và
- `prices[j] <= prices[i]` (giá **nhỏ hơn hoặc bằng**).

Nếu không có `j` như vậy thì **không được giảm**.

Trả về mảng `answer` với `answer[i]` là giá cuối cùng phải trả cho món `i`.

| `prices` | Output | Giải thích ngắn |
|---|---|---|
| `[8,4,6,2,3]` | `[4,2,4,2,3]` | 8 gặp 4 → 4; 4 gặp 2 → 2; 6 gặp 2 → 4; 2 và 3 không ai phía sau ≤ |
| `[1,2,3,4,5]` | `[1,2,3,4,5]` | dãy tăng: không món nào phía sau rẻ hơn hoặc bằng |
| `[10,1,1,6]` | `[9,0,1,6]` | 10 gặp 1 → 9; 1 gặp 1 (bằng nhau vẫn tính) → 0 |

**Constraints:** `1 <= prices.length <= 500`, `1 <= prices[i] <= 1000`.

Câu hỏi đặt ra trước khi code:
- "Chỉ số nhỏ nhất `j > i`" nghĩa là gì — món **gần nhất** bên phải hay món **rẻ nhất** bên phải?
- Bằng nhau (`<=`) có được tính không? (Có — xem ví dụ 3.)
- Có cách nào không phải quét lại toàn bộ phần bên phải cho mỗi món?

### 1.1. Hiểu đúng chữ "nhỏ nhất" — gần nhất, KHÔNG phải rẻ nhất

"`j` nhỏ nhất" là **chỉ số** nhỏ nhất, tức món **đầu tiên** gặp khi đi từ `i+1` sang phải mà giá `<= prices[i]`.

Ví dụ `prices = [8, 4, 6, 2, 3]`, món `i = 0` giá 8:

| j | prices[j] | `prices[j] <= 8`? |
|---:|---:|:---:|
| 1 | 4 | ✅ dừng ngay ở đây |
| 3 | 2 | (rẻ hơn nữa nhưng không xét) |

Nên giảm 4 chứ không phải giảm 2. Kết quả `8 - 4 = 4`.

Cell dưới đây chỉ in ra các món phía sau món 0 để ta nhìn bằng mắt.

In [1]:
prices = [8, 4, 6, 2, 3]
i = 0
print("Món", i, "giá", prices[i])
for j in range(i + 1, len(prices)):
    ok = prices[j] <= prices[i]
    print(f"  j={j}, prices[j]={prices[j]}, <= {prices[i]}? {ok}")

Món 0 giá 8
  j=1, prices[j]=4, <= 8? True
  j=2, prices[j]=6, <= 8? True
  j=3, prices[j]=2, <= 8? True
  j=4, prices[j]=3, <= 8? True


Nhìn output: `j=1` đã thỏa, nên **món đầu tiên thỏa** là `j=1`. Các `j` sau cũng thỏa nhưng không quan trọng — đề chỉ lấy món gần nhất.

## 2. Nền tảng Python cần biết

### 2.1. `range(a, b)` và duyệt "phần bên phải"

`range(i + 1, n)` sinh các số `i+1, i+2, ..., n-1` — đúng là "tất cả vị trí nằm sau `i`". Nếu `i = n - 1` thì `range(n, n)` **rỗng**, vòng lặp không chạy lần nào (món cuối không bao giờ được giảm).

In [2]:
n = 5
for i in [0, 3, 4]:
    print(f"i={i} -> các j phía sau:", list(range(i + 1, n)))

i=0 -> các j phía sau: [1, 2, 3, 4]
i=3 -> các j phía sau: [4]
i=4 -> các j phía sau: []


### 2.2. `break` — thoát vòng lặp ngay khi tìm thấy

Khi tìm được `j` đầu tiên thỏa điều kiện, ta không cần xét tiếp. `break` thoát khỏi vòng lặp **gần nhất** chứa nó.

In [3]:
for x in [5, 7, 2, 9, 1]:
    print("xét", x)
    if x < 3:
        print("  gặp số < 3 đầu tiên:", x, "-> break")
        break
print("đã ra khỏi vòng lặp")

xét 5
xét 7
xét 2
  gặp số < 3 đầu tiên: 2 -> break
đã ra khỏi vòng lặp


### 2.3. Sao chép list: `prices[:]` và `list(prices)`

Đề trả về một mảng **mới**. Cách gọn là tạo bản sao `answer = prices[:]` rồi trừ dần. Nếu viết `answer = prices` thì hai tên trỏ **cùng một list** — sửa `answer` sẽ sửa luôn `prices` (có thể làm hỏng dữ liệu của người gọi).

In [4]:
a = [8, 4, 6]
alias = a          # cùng một đối tượng
copy = a[:]        # bản sao mới
alias[0] = 100
print("a     =", a)       # bị đổi theo alias
print("copy  =", copy)    # giữ nguyên
print("alias is a:", alias is a, "| copy is a:", copy is a)

a     = [100, 4, 6]
copy  = [8, 4, 6]
alias is a: True | copy is a: False


### 2.4. Stack (ngăn xếp) bằng list

**Stack** là cấu trúc "vào sau ra trước" (LIFO — Last In, First Out), giống chồng đĩa: đĩa đặt sau cùng nằm trên cùng và được lấy ra trước.

Trong Python dùng `list`:

| Thao tác | Cú pháp | Độ phức tạp |
|---|---|---|
| Đẩy vào đỉnh (push) | `stack.append(x)` | `O(1)` trung bình (amortized) |
| Xem đỉnh (peek) | `stack[-1]` | `O(1)` |
| Lấy ra khỏi đỉnh (pop) | `stack.pop()` | `O(1)` |
| Kiểm tra rỗng | `if stack:` / `while stack:` | `O(1)` |

`stack[-1]` là phần tử **cuối** list = đỉnh stack. Lưu ý: gọi `stack[-1]` hay `stack.pop()` khi stack rỗng sẽ báo `IndexError`, nên luôn kiểm tra `stack` khác rỗng trước.

In [5]:
stack = []
stack.append(1)
stack.append(2)
stack.append(3)
print("stack:", stack, "| đỉnh:", stack[-1])
top = stack.pop()
print("pop ra:", top, "| còn lại:", stack)
print("stack rỗng?", not stack)
while stack:
    print("pop:", stack.pop())
print("stack rỗng?", not stack)

stack: [1, 2, 3] | đỉnh: 3
pop ra: 3 | còn lại: [1, 2]
stack rỗng? False
pop: 2
pop: 1
stack rỗng? True


## 3. Cách 1 — Brute force: với mỗi món, quét sang phải

Đây chính là gợi ý của đề: với món `i`, chạy `j` từ `i+1`, gặp món đầu tiên `prices[j] <= prices[i]` thì trừ và `break`.

Từng đoạn:
1. `answer = prices[:]` — mặc định không giảm.
2. Vòng ngoài `for i in range(n)`.
3. Vòng trong `for j in range(i + 1, n)`; nếu thỏa thì `answer[i] -= prices[j]` rồi `break`.

In [6]:
def final_prices_bruteforce(prices: list[int]) -> list[int]:
    n = len(prices)
    answer = prices[:]                 # mặc định: không giảm
    for i in range(n):
        for j in range(i + 1, n):
            if prices[j] <= prices[i]:
                answer[i] = prices[i] - prices[j]
                break                  # chỉ lấy món GẦN NHẤT
    return answer

print(final_prices_bruteforce([8, 4, 6, 2, 3]))   # [4, 2, 4, 2, 3]
print(final_prices_bruteforce([1, 2, 3, 4, 5]))   # [1, 2, 3, 4, 5]
print(final_prices_bruteforce([10, 1, 1, 6]))     # [9, 0, 1, 6]

[4, 2, 4, 2, 3]
[1, 2, 3, 4, 5]
[9, 0, 1, 6]


### Code nộp LeetCode — Brute force

Với `n <= 500`, số phép so sánh tối đa khoảng `500*499/2 ≈ 125.000` — chạy rất nhanh, nộp được.

In [7]:
class Solution:
    def finalPrices(self, prices: list[int]) -> list[int]:
        n = len(prices)
        answer = prices[:]
        for i in range(n):
            for j in range(i + 1, n):
                if prices[j] <= prices[i]:
                    answer[i] -= prices[j]
                    break
        return answer

print(Solution().finalPrices([8, 4, 6, 2, 3]))

[4, 2, 4, 2, 3]


**Độ phức tạp brute force** (`n = len(prices)`):
- Thời gian: `O(n²)` trong trường hợp xấu nhất — ví dụ dãy **tăng dần** `[1,2,3,...]`: không món nào gặp được giá ≤, vòng trong luôn chạy tới cuối.
- Bộ nhớ: `O(1)` phụ trợ (không tính mảng `answer` phải trả về).

## 4. Nút thắt — ta đang lặp lại công việc gì?

Đếm số lần so sánh của brute force trên dãy tăng dần:

In [8]:
def count_comparisons(prices):
    n = len(prices)
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            cnt += 1
            if prices[j] <= prices[i]:
                break
    return cnt

for n in [5, 50, 500]:
    inc = list(range(1, n + 1))
    print(f"n={n:>3}: dãy tăng -> {count_comparisons(inc):>6} phép so sánh (= n(n-1)/2 = {n*(n-1)//2})")

n=  5: dãy tăng ->     10 phép so sánh (= n(n-1)/2 = 10)
n= 50: dãy tăng ->   1225 phép so sánh (= n(n-1)/2 = 1225)
n=500: dãy tăng -> 124750 phép so sánh (= n(n-1)/2 = 124750)


Vấn đề: khi xét món `i`, ta quét lại những món bên phải mà các món trước đó đã quét rồi. Thông tin "món nào còn đang **chờ** được giảm" bị vứt đi sau mỗi vòng ngoài.

Đổi góc nhìn: thay vì **mỗi món đi tìm người giảm giá cho mình**, hãy để **mỗi món mới xuất hiện "trả lời" cho các món cũ đang chờ**.

## 5. Insight — stack các món "đang chờ giảm giá"

Duyệt từ trái sang phải. Giữ một stack chứa **chỉ số** các món chưa tìm được người giảm giá.

Khi gặp món `j` giá `p = prices[j]`:
- Nếu món ở đỉnh stack có giá `>= p`, thì `j` chính là món **đầu tiên** bên phải nó có giá `<=` nó → giảm giá cho món đó, `pop` ra.
- Lặp lại cho tới khi stack rỗng hoặc đỉnh có giá `< p`.
- Đẩy `j` vào stack (bây giờ chính `j` đang chờ).

**Quan sát quan trọng:** các giá trong stack (từ đáy lên đỉnh) luôn **tăng dần nghiêm ngặt**. Vì nếu đỉnh ≥ p thì nó đã bị pop. Do đó khi đỉnh `< p`, mọi phần tử bên dưới còn nhỏ hơn nữa → không ai bên dưới thỏa → dừng `while` được ngay. Đây là lý do gọi là **monotonic stack** (stack đơn điệu).

Ví dụ nhỏ: thêm lần lượt `8, 4` vào stack.

In [9]:
prices = [8, 4]
answer = prices[:]
stack = []
# j = 0, giá 8: stack rỗng -> đẩy vào
stack.append(0)
print("sau j=0:", stack)
# j = 1, giá 4: đỉnh là chỉ số 0 có giá 8 >= 4 -> món 0 được giảm 4
top = stack.pop()
answer[top] -= prices[1]
stack.append(1)
print("sau j=1:", stack, "| answer =", answer)

sau j=0: [0]
sau j=1: [1] | answer = [4, 4]


## 6. Cách 2 — Monotonic stack: mổ xẻ từng đoạn rồi ghép

**Đoạn A — khởi tạo**
```python
answer = prices[:]   # món nào không bao giờ bị pop thì giữ nguyên giá
stack = []           # chứa CHỈ SỐ, không phải giá
```

**Đoạn B — với mỗi món `j`, giải quyết các món đang chờ**
```python
while stack and prices[stack[-1]] >= prices[j]:
    i = stack.pop()
    answer[i] -= prices[j]
```
- `stack and ...` dùng **đánh giá ngắn mạch** (short-circuit): nếu `stack` rỗng thì Python không tính vế sau → không bị `IndexError`.
- Dùng `>=` vì đề cho phép giá **bằng** (`prices[j] <= prices[i]`).

**Đoạn C — món `j` bắt đầu chờ**
```python
stack.append(j)
```

Vì sao lưu **chỉ số**? Vì cần biết ghi kết quả vào `answer[i]` nào. Giá thì tra lại bằng `prices[i]`.

In [10]:
def final_prices_stack(prices: list[int]) -> list[int]:
    answer = prices[:]
    stack = []                                   # chỉ số các món đang chờ, giá tăng dần
    for j, p in enumerate(prices):
        while stack and prices[stack[-1]] >= p:  # đỉnh tìm được món giảm giá: chính là j
            i = stack.pop()
            answer[i] -= p
        stack.append(j)
    return answer

print(final_prices_stack([8, 4, 6, 2, 3]))   # [4, 2, 4, 2, 3]
print(final_prices_stack([1, 2, 3, 4, 5]))   # [1, 2, 3, 4, 5]
print(final_prices_stack([10, 1, 1, 6]))     # [9, 0, 1, 6]

[4, 2, 4, 2, 3]
[1, 2, 3, 4, 5]
[9, 0, 1, 6]


### 6.1. `enumerate` — lấy cả chỉ số lẫn giá trị

`for j, p in enumerate(prices)` cho từng cặp `(chỉ số, giá trị)`, gọn hơn `for j in range(len(prices)): p = prices[j]`.

In [11]:
for j, p in enumerate([8, 4, 6]):
    print("j =", j, "| p =", p)

j = 0 | p = 8
j = 1 | p = 4
j = 2 | p = 6


## 7. TRACE từng bước — nhìn stack thay đổi

Ví dụ `prices = [8, 4, 6, 2, 3]`. Cột "stack" ghi dạng `chỉ số(giá)`.

| j | p | stack trước | Hành động | stack sau | answer |
|---:|---:|---|---|---|---|
| 0 | 8 | `[]` | rỗng → đẩy 0 | `[0(8)]` | `[8,4,6,2,3]` |
| 1 | 4 | `[0(8)]` | 8 ≥ 4 → pop 0, answer[0]=8-4=4; đẩy 1 | `[1(4)]` | `[4,4,6,2,3]` |
| 2 | 6 | `[1(4)]` | 4 < 6 → không pop; đẩy 2 | `[1(4),2(6)]` | `[4,4,6,2,3]` |
| 3 | 2 | `[1(4),2(6)]` | 6 ≥ 2 → pop 2, answer[2]=4; 4 ≥ 2 → pop 1, answer[1]=2; đẩy 3 | `[3(2)]` | `[4,2,4,2,3]` |
| 4 | 3 | `[3(2)]` | 2 < 3 → không pop; đẩy 4 | `[3(2),4(3)]` | `[4,2,4,2,3]` |

Hết mảng: món 3 và 4 vẫn trong stack → không được giảm, giữ nguyên giá. Kết quả `[4,2,4,2,3]` ✔.

Để ý ở `j = 3`: **một** món mới giải quyết **hai** món cũ cùng lúc — đây là chỗ stack tiết kiệm công.

In [12]:
def trace_final_prices(prices):
    answer = prices[:]
    stack = []
    show = lambda st: "[" + ", ".join(f"{k}({prices[k]})" for k in st) + "]"
    for j, p in enumerate(prices):
        print(f"j={j}, p={p}, stack trước = {show(stack)}")
        while stack and prices[stack[-1]] >= p:
            i = stack.pop()
            answer[i] -= p
            print(f"    giá {prices[i]} (món {i}) >= {p} -> pop {i}, answer[{i}] = {prices[i]} - {p} = {answer[i]}")
        stack.append(j)
        print(f"    đẩy {j} -> stack sau = {show(stack)}, answer = {answer}")
    print("Còn trong stack (không được giảm):", show(stack))
    return answer

trace_final_prices([8, 4, 6, 2, 3])

j=0, p=8, stack trước = []
    đẩy 0 -> stack sau = [0(8)], answer = [8, 4, 6, 2, 3]
j=1, p=4, stack trước = [0(8)]
    giá 8 (món 0) >= 4 -> pop 0, answer[0] = 8 - 4 = 4
    đẩy 1 -> stack sau = [1(4)], answer = [4, 4, 6, 2, 3]
j=2, p=6, stack trước = [1(4)]
    đẩy 2 -> stack sau = [1(4), 2(6)], answer = [4, 4, 6, 2, 3]
j=3, p=2, stack trước = [1(4), 2(6)]
    giá 6 (món 2) >= 2 -> pop 2, answer[2] = 6 - 2 = 4
    giá 4 (món 1) >= 2 -> pop 1, answer[1] = 4 - 2 = 2
    đẩy 3 -> stack sau = [3(2)], answer = [4, 2, 4, 2, 3]
j=4, p=3, stack trước = [3(2)]
    đẩy 4 -> stack sau = [3(2), 4(3)], answer = [4, 2, 4, 2, 3]
Còn trong stack (không được giảm): [3(2), 4(3)]


[4, 2, 4, 2, 3]

Trace thêm ví dụ có **giá bằng nhau** `[10, 1, 1, 6]` để thấy vai trò của `>=`:

In [13]:
trace_final_prices([10, 1, 1, 6])

j=0, p=10, stack trước = []
    đẩy 0 -> stack sau = [0(10)], answer = [10, 1, 1, 6]
j=1, p=1, stack trước = [0(10)]
    giá 10 (món 0) >= 1 -> pop 0, answer[0] = 10 - 1 = 9
    đẩy 1 -> stack sau = [1(1)], answer = [9, 1, 1, 6]
j=2, p=1, stack trước = [1(1)]
    giá 1 (món 1) >= 1 -> pop 1, answer[1] = 1 - 1 = 0
    đẩy 2 -> stack sau = [2(1)], answer = [9, 0, 1, 6]
j=3, p=6, stack trước = [2(1)]
    đẩy 3 -> stack sau = [2(1), 3(6)], answer = [9, 0, 1, 6]
Còn trong stack (không được giảm): [2(1), 3(6)]


[9, 0, 1, 6]

Ở `j = 2` (giá 1), đỉnh là món 1 cũng giá 1. Vì `1 >= 1` nên món 1 được giảm 1 → `answer[1] = 0`. Nếu viết nhầm `>` thì món 1 không được giảm — sai so với đề.

## 8. Cách 3 — Duyệt từ phải sang trái (stack "ứng viên giảm giá")

Một góc nhìn khác, cũng `O(n)`: đi từ phải sang trái, stack giữ các **giá ứng viên** có thể làm người giảm cho món đang xét.

Với món `i` giá `p`:
1. Pop mọi giá `> p` trên đỉnh: chúng không giảm được cho `p`, và cũng **vô dụng** cho các món bên trái (vì `p` nằm gần hơn và rẻ hơn hoặc bằng chúng — món bên trái nào chịu được chúng thì cũng chịu được `p`, mà `p` gần hơn).
2. Nếu còn đỉnh thì đó là giá ≤ p gần nhất bên phải → `answer[i] = p - đỉnh`.
3. Đẩy `p` vào.

In [14]:
def final_prices_right_to_left(prices: list[int]) -> list[int]:
    n = len(prices)
    answer = [0] * n
    stack = []                          # các giá ứng viên; từ đáy lên đỉnh: không giảm
    for i in range(n - 1, -1, -1):
        p = prices[i]
        while stack and stack[-1] > p:  # quá đắt, không giảm được cho p và bị p che khuất
            stack.pop()
        answer[i] = p - stack[-1] if stack else p
        stack.append(p)
    return answer

print(final_prices_right_to_left([8, 4, 6, 2, 3]))
print(final_prices_right_to_left([10, 1, 1, 6]))

[4, 2, 4, 2, 3]
[9, 0, 1, 6]


### 8.1. `range(n - 1, -1, -1)` — đếm ngược

`range(start, stop, step)` với `step = -1` đi lùi, dừng **trước** `stop`. Muốn đi tới chỉ số 0 thì `stop` phải là `-1`.

Còn biểu thức `a if điều_kiện else b` là **toán tử ba ngôi** (conditional expression): nếu điều kiện đúng lấy `a`, ngược lại lấy `b`.

In [15]:
print(list(range(4, -1, -1)))     # [4, 3, 2, 1, 0]
print(list(range(4, 0, -1)))      # [4, 3, 2, 1]  <- thiếu 0!
stack = []
print("rỗng ->", 7 - stack[-1] if stack else 7)
stack = [3]
print("có 3 ->", 7 - stack[-1] if stack else 7)

[4, 3, 2, 1, 0]
[4, 3, 2, 1]
rỗng -> 7
có 3 -> 4


## 9. Vì sao thuật toán stack (cách 2) đúng?

**Invariant (bất biến)** — ngay trước khi xử lý món `j`:
1. Stack chứa đúng các chỉ số `i < j` mà **chưa có** món nào trong khoảng `(i, j)` có giá `<= prices[i]`.
2. Giá theo thứ tự trong stack (đáy → đỉnh) **tăng nghiêm ngặt**.

Khi xử lý `j`:
- Món `i` ở đỉnh có `prices[i] >= prices[j]`: theo (1), không có món nào giữa `i` và `j` giảm được cho `i`, nên `j` là chỉ số **nhỏ nhất** thỏa → giảm đúng. Pop `i` vì nó đã xong.
- Khi đỉnh có giá `< prices[j]`: theo (2), mọi phần tử bên dưới còn nhỏ hơn → không phần tử nào thỏa → dừng là đúng, không bỏ sót.
- Đẩy `j`: (1) đúng cho `j` (chưa có ai sau nó); (2) vẫn giữ vì đỉnh cũ `< prices[j]`.

Kết thúc: những món còn trong stack không bao giờ gặp được giá ≤ ở bên phải → giữ nguyên giá, đúng như đề.

## 10. Code nộp LeetCode — monotonic stack

In [16]:
class Solution:
    def finalPrices(self, prices: list[int]) -> list[int]:
        answer = prices[:]
        stack = []
        for j, p in enumerate(prices):
            while stack and prices[stack[-1]] >= p:
                answer[stack.pop()] -= p
            stack.append(j)
        return answer

sol = Solution()
assert sol.finalPrices([8, 4, 6, 2, 3]) == [4, 2, 4, 2, 3]
assert sol.finalPrices([1, 2, 3, 4, 5]) == [1, 2, 3, 4, 5]
assert sol.finalPrices([10, 1, 1, 6]) == [9, 0, 1, 6]
print("Solution (stack) PASS 3 ví dụ của đề")

Solution (stack) PASS 3 ví dụ của đề


## 11. Phân tích Time / Space

Gọi `n = len(prices)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` worst | `O(1)` | dễ nghĩ nhất, đủ nhanh với n ≤ 500 |
| 2. Stack trái → phải | `O(n)` | `O(n)` | mỗi chỉ số **push 1 lần, pop tối đa 1 lần** |
| 3. Stack phải → trái | `O(n)` | `O(n)` | stack giữ giá ứng viên |

Vì sao cách 2 là `O(n)` dù có `while` lồng trong `for`? Hãy đếm **tổng** số lần pop trong cả chương trình: mỗi chỉ số được đẩy vào đúng 1 lần nên bị pop **tối đa** 1 lần → tổng số lần chạy thân `while` ≤ `n`. Cộng với `n` lần lặp `for` → `O(2n) = O(n)`. Cách đếm này gọi là **phân tích khấu hao** (amortized analysis).

Kiểm chứng bằng cách đếm:

In [17]:
def count_stack_ops(prices):
    stack, pops = [], 0
    for j, p in enumerate(prices):
        while stack and prices[stack[-1]] >= p:
            stack.pop(); pops += 1
        stack.append(j)
    return pops

import random
random.seed(1)
for n in [10, 100, 500]:
    arr = [random.randint(1, 1000) for _ in range(n)]
    print(f"n={n:>3}: tổng pop = {count_stack_ops(arr):>3} (luôn <= n), brute force so sánh = {count_comparisons(arr)}")

n= 10: tổng pop =   6 (luôn <= n), brute force so sánh = 20
n=100: tổng pop =  91 (luôn <= n), brute force so sánh = 482
n=500: tổng pop = 496 (luôn <= n), brute force so sánh = 3027


## 12. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ đề.
2. Biên: 1 phần tử; toàn bộ bằng nhau; dãy giảm; dãy tăng.
3. Dễ sai: giá bằng nhau (`>=` vs `>`), món gần nhất không phải rẻ nhất.
4. Đối chiếu 3 cách trên nhiều mảng ngẫu nhiên.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<32} -> {actual}")

tests = [
    ("đề 1",                     [8, 4, 6, 2, 3], [4, 2, 4, 2, 3]),
    ("đề 2 (tăng dần)",          [1, 2, 3, 4, 5], [1, 2, 3, 4, 5]),
    ("đề 3 (giá bằng nhau)",     [10, 1, 1, 6],   [9, 0, 1, 6]),
    ("1 phần tử",                [7],             [7]),
    ("tất cả bằng nhau",         [5, 5, 5],       [0, 0, 5]),
    ("giảm dần",                 [5, 4, 3, 2, 1], [1, 1, 1, 1, 1]),
    ("gần nhất, không rẻ nhất",  [9, 8, 1],       [1, 7, 1]),
    ("giá tối đa",               [1000, 1000],    [0, 1000]),
]
for name, arr, exp in tests:
    for f in (final_prices_bruteforce, final_prices_stack, final_prices_right_to_left):
        assert f(arr) == exp, (f.__name__, arr)
    check(name, Solution().finalPrices(arr), exp)

PASS  đề 1                             -> [4, 2, 4, 2, 3]
PASS  đề 2 (tăng dần)                  -> [1, 2, 3, 4, 5]
PASS  đề 3 (giá bằng nhau)             -> [9, 0, 1, 6]
PASS  1 phần tử                        -> [7]
PASS  tất cả bằng nhau                 -> [0, 0, 5]
PASS  giảm dần                         -> [1, 1, 1, 1, 1]
PASS  gần nhất, không rẻ nhất          -> [1, 7, 1]
PASS  giá tối đa                       -> [0, 1000]


In [19]:
# Đối chiếu ngẫu nhiên: 3 cách phải luôn cho cùng kết quả
random.seed(2026)
for _ in range(2000):
    n = random.randint(1, 30)
    arr = [random.randint(1, 6) for _ in range(n)]   # giá nhỏ -> nhiều giá trùng, dễ lộ lỗi >= / >
    a = final_prices_bruteforce(arr)
    assert final_prices_stack(arr) == a, arr
    assert final_prices_right_to_left(arr) == a, arr
# Kiểm tra không làm hỏng input
orig = [8, 4, 6, 2, 3]
keep = orig[:]
Solution().finalPrices(orig)
assert orig == keep
print("2000 mảng ngẫu nhiên: 3 cách khớp nhau; input không bị sửa. PASS")

2000 mảng ngẫu nhiên: 3 cách khớp nhau; input không bị sửa. PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>` thay vì `>=` trong stack

```python
while stack and prices[stack[-1]] > p:   # SAI
```
Vì sao sai: đề cho giảm khi `prices[j] <= prices[i]`, tức cả trường hợp **bằng**. Ví dụ `[10,1,1,6]` sẽ ra `[9,1,1,6]` thay vì `[9,0,1,6]`.

### Lỗi 2: Lấy món **rẻ nhất** bên phải thay vì món **gần nhất**

```python
answer[i] = prices[i] - min(x for x in prices[i+1:] if x <= prices[i])   # SAI
```
Với `[9,8,1]`, món 9 sẽ bị trừ 1 thay vì 8.

### Lỗi 3: Quên `break` trong brute force — trừ nhiều lần

### Lỗi 4: `answer = prices` (không sao chép) rồi sửa `answer` → làm hỏng `prices` của người gọi. Kết quả trả về có thể vẫn đúng (vì ta chỉ sửa `answer[i]` sau khi đã dùng xong `prices[i]`), nhưng input bị thay đổi là lỗi "tác dụng phụ" (side effect) rất khó dò trong chương trình thật.

Chạy để thấy từng lỗi:

In [20]:
def wrong_strict(prices):
    answer, stack = prices[:], []
    for j, p in enumerate(prices):
        while stack and prices[stack[-1]] > p:     # lỗi 1
            answer[stack.pop()] -= p
        stack.append(j)
    return answer

def wrong_cheapest(prices):
    ans = []
    for i, p in enumerate(prices):
        cand = [x for x in prices[i+1:] if x <= p]
        ans.append(p - min(cand) if cand else p)   # lỗi 2
    return ans

def wrong_no_break(prices):
    answer = prices[:]
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            if prices[j] <= prices[i]:
                answer[i] -= prices[j]             # lỗi 3: thiếu break
    return answer

def wrong_alias(prices):
    answer = prices                                # lỗi 4: không copy
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            if prices[j] <= prices[i]:
                answer[i] -= prices[j]
                break
    return answer

print("lỗi 1 [10,1,1,6]:", wrong_strict([10, 1, 1, 6]), "| đúng:", final_prices_stack([10, 1, 1, 6]))
print("lỗi 2 [9,8,1]   :", wrong_cheapest([9, 8, 1]), "| đúng:", final_prices_stack([9, 8, 1]))
print("lỗi 3 [8,4,6,2,3]:", wrong_no_break([8, 4, 6, 2, 3]), "| đúng:", final_prices_stack([8, 4, 6, 2, 3]))
data = [4, 4, 4]
print("lỗi 4 [4,4,4]   :", wrong_alias(data), "| đúng:", final_prices_stack([4, 4, 4]), "| input bị đổi thành", data)

lỗi 1 [10,1,1,6]: [9, 1, 1, 6] | đúng: [9, 0, 1, 6]
lỗi 2 [9,8,1]   : [8, 7, 1] | đúng: [1, 7, 1]
lỗi 3 [8,4,6,2,3]: [-7, -1, 1, 2, 3] | đúng: [4, 2, 4, 2, 3]
lỗi 4 [4,4,4]   : [0, 0, 4] | đúng: [0, 0, 4] | input bị đổi thành [0, 0, 4]


Ở lỗi 4: kết quả trùng đáp án đúng, nhưng list `data` của người gọi đã bị đổi từ `[4, 4, 4]` thành `[0, 0, 4]`. Nếu sau đó chương trình dùng lại `data` (ví dụ tính tổng giá gốc) sẽ sai mà không hiểu vì sao.

### Lỗi 5: Gọi `stack[-1]` khi stack rỗng

```python
while prices[stack[-1]] >= p and stack:   # SAI thứ tự: tính stack[-1] trước khi kiểm tra rỗng
```
Phải đặt `stack` **trước** để short-circuit bảo vệ.

In [21]:
stack = []
try:
    while prices[stack[-1]] >= 1 and stack:
        pass
except IndexError as e:
    print("IndexError:", e)
while stack and prices[stack[-1]] >= 1:   # đúng thứ tự: không lỗi
    pass
print("Thứ tự đúng: không lỗi")

IndexError: list index out of range
Thứ tự đúng: không lỗi


## 14. Mẹo nhận dạng pattern

Gặp các cụm từ sau → nghĩ ngay tới **monotonic stack**:
- "phần tử **đầu tiên / gần nhất** bên phải (hoặc trái) **nhỏ hơn / lớn hơn**…"
- "next greater element", "next smaller element", "previous smaller", "số ngày phải chờ tới khi ấm hơn"…

Quy tắc chọn chiều so sánh (duyệt trái → phải, stack chứa các phần tử đang chờ):

| Muốn tìm phía phải phần tử… | Điều kiện pop | Stack (đáy → đỉnh) |
|---|---|---|
| nhỏ hơn hoặc bằng (bài này) | `prices[top] >= x` | tăng nghiêm ngặt |
| nhỏ hơn nghiêm ngặt | `a[top] > x` | không giảm |
| lớn hơn nghiêm ngặt | `a[top] < x` | không tăng |
| lớn hơn hoặc bằng | `a[top] <= x` | giảm nghiêm ngặt |

Mẹo nhớ: điều kiện pop chính là "**phần tử mới x là đáp án của đỉnh**".

## 15. Biến thể

### Biến thể A — trả về **chỉ số** món giảm giá (hoặc -1)

In [22]:
def next_smaller_or_equal_index(a):
    res = [-1] * len(a)
    stack = []
    for j, x in enumerate(a):
        while stack and a[stack[-1]] >= x:
            res[stack.pop()] = j
        stack.append(j)
    return res

print(next_smaller_or_equal_index([8, 4, 6, 2, 3]))   # [1, 3, 3, -1, -1]

[1, 3, 3, -1, -1]


### Biến thể B — Next Greater Element (giống LeetCode 496/503/739)

Với mỗi phần tử, tìm giá trị **lớn hơn nghiêm ngặt** gần nhất bên phải, không có thì `-1`. Chỉ cần đổi điều kiện pop thành `a[top] < x`.

In [23]:
def next_greater(a):
    res = [-1] * len(a)
    stack = []
    for j, x in enumerate(a):
        while stack and a[stack[-1]] < x:
            res[stack.pop()] = x
        stack.append(j)
    return res

print(next_greater([2, 1, 2, 4, 3]))   # [4, 2, 4, -1, -1]

[4, 2, 4, -1, -1]


### Biến thể C — Daily Temperatures (LeetCode 739): số ngày phải chờ

Thay vì giá trị, lưu **khoảng cách** `j - i`. Đây là bài tập 5 ở dưới, bạn sẽ tự viết.

## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (đoán output)
Không chạy code, hãy đoán `final_prices_stack([3, 3, 3, 1])`. Sau đó tự viết hàm `exercise_predict()` trả về list bạn đoán.

In [24]:
def exercise_predict() -> list[int]:
    # TODO: trả về list bạn đoán cho final_prices_stack([3, 3, 3, 1])
    pass

# assert exercise_predict() == final_prices_stack([3, 3, 3, 1])

### Bài 2 (tính tay)
Lập bảng trace stack cho `prices = [5, 7, 3, 8, 3]`. Ghi kết quả cuối vào hàm.

In [25]:
def exercise_trace_result() -> list[int]:
    # TODO: tự lập bảng j | p | stack trước | hành động | stack sau, rồi trả về answer
    pass

# assert exercise_trace_result() == [2, 4, 0, 5, 3]

### Bài 3 (điền chỗ trống)
Hoàn thiện lời giải stack.

In [26]:
def exercise_final_prices(prices: list[int]) -> list[int]:
    answer = prices[:]
    stack = []
    for j, p in enumerate(prices):
        # TODO 1: while stack khác rỗng VÀ giá ở đỉnh >= p
        # TODO 2:     pop chỉ số i, trừ p vào answer[i]
        # TODO 3: đẩy j vào stack
        pass
    return answer

# assert exercise_final_prices([8, 4, 6, 2, 3]) == [4, 2, 4, 2, 3]
# assert exercise_final_prices([10, 1, 1, 6]) == [9, 0, 1, 6]

### Bài 4 (biến thể — phía trái)
Với mỗi món `i`, tìm món **gần nhất bên trái** (`j < i`) có giá **nhỏ hơn nghiêm ngặt**; trả về giá đó, không có thì `-1`.

Ví dụ `[3, 7, 8, 4]` → `[-1, 3, 7, 3]`.

In [27]:
def exercise_previous_smaller(a: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_previous_smaller([3, 7, 8, 4]) == [-1, 3, 7, 3]
# assert exercise_previous_smaller([1, 1, 1]) == [-1, -1, -1]

### Bài 5 (Daily Temperatures)
Cho `temps`, trả về `wait[i]` = số ngày phải chờ tới ngày đầu tiên **ấm hơn nghiêm ngặt**; không có thì 0.

Ví dụ `[73,74,75,71,69,72,76,73]` → `[1,1,4,2,1,1,0,0]`.

In [28]:
def exercise_daily_temperatures(temps: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_daily_temperatures([73,74,75,71,69,72,76,73]) == [1,1,4,2,1,1,0,0]
# assert exercise_daily_temperatures([30, 60, 90]) == [1, 1, 0]

### Bài 6 (mini challenge — vòng tròn)
Cửa hàng xếp thành **vòng tròn**: sau món cuối quay về món 0. Món `i` được giảm bằng giá món **gần nhất theo chiều kim đồng hồ** (không tính chính nó) có giá `<=`. Trả về giá cuối.

Ví dụ `[4, 6, 2, 3]` → món 3 (giá 3) đi vòng gặp 4, 6 rồi 2 ≤ 3 → `3 - 2 = 1`. Kết quả `[2, 4, 2, 1]`. Món giá 2 đi vòng: 3, 4, 6 đều > 2 → không giảm.

In [29]:
def exercise_circular_final_prices(prices: list[int]) -> list[int]:
    # TODO: gợi ý — duyệt 2n bước, chỉ số thật là k % n
    pass

# assert exercise_circular_final_prices([4, 6, 2, 3]) == [2, 4, 2, 1]
# assert exercise_circular_final_prices([5, 5]) == [0, 0]

### Gợi ý ba mức

**Bài 1–2**
- Mức 1: nhớ `>=` — giá bằng nhau vẫn giảm.
- Mức 2: viết stack dạng `chỉ số(giá)` mỗi bước.
- Mức 3: ở bài 2, `j=2` (giá 3) pop cả món 1 (giá 7) và món 0 (giá 5).

**Bài 3**
- Mức 1: ba dòng code giống hệt mục 6.
- Mức 2: `while stack and prices[stack[-1]] >= p:`
- Mức 3: `answer[stack.pop()] -= p` rồi `stack.append(j)`.

**Bài 4**
- Mức 1: duyệt trái → phải, stack chứa các **ứng viên** ở bên trái.
- Mức 2: trước khi trả lời cho `x`, pop mọi phần tử `>= x` (chúng không nhỏ hơn x, và x che khuất chúng cho các phần tử sau).
- Mức 3: đáp án là `stack[-1]` nếu stack khác rỗng, rồi `append(x)`.

**Bài 5**
- Mức 1: là "next greater" — điều kiện pop `temps[top] < t`.
- Mức 2: stack lưu **chỉ số** để tính khoảng cách.
- Mức 3: `wait[i] = j - i` khi pop `i` bởi `j`.

**Bài 6**
- Mức 1: duyệt mảng **hai lần** để các món cuối "nhìn thấy" phần đầu.
- Mức 2: `for k in range(2 * n): j = k % n`; chỉ `append` khi `k < n`.
- Mức 3: ở vòng 2, khi gặp lại chính món `j` đang nằm ở đỉnh, đừng để nó tự giảm cho chính nó: thêm điều kiện `stack[-1] != j` vào `while`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [30]:
def answer_predict():
    return [0, 0, 2, 1]

def answer_trace_result():
    return [2, 4, 0, 5, 3]

def answer_final_prices(prices):
    answer, stack = prices[:], []
    for j, p in enumerate(prices):
        while stack and prices[stack[-1]] >= p:
            answer[stack.pop()] -= p
        stack.append(j)
    return answer

def answer_previous_smaller(a):
    res, stack = [], []
    for x in a:
        while stack and stack[-1] >= x:
            stack.pop()
        res.append(stack[-1] if stack else -1)
        stack.append(x)
    return res

def answer_daily_temperatures(temps):
    wait, stack = [0] * len(temps), []
    for j, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            i = stack.pop()
            wait[i] = j - i
        stack.append(j)
    return wait

def answer_circular_final_prices(prices):
    n = len(prices)
    answer, stack = prices[:], []
    for k in range(2 * n):
        j = k % n
        while stack and stack[-1] != j and prices[stack[-1]] >= prices[j]:
            answer[stack.pop()] -= prices[j]
        if k < n:
            stack.append(j)
    return answer

assert answer_predict() == final_prices_stack([3, 3, 3, 1])
assert answer_trace_result() == final_prices_stack([5, 7, 3, 8, 3])
assert answer_final_prices([8, 4, 6, 2, 3]) == [4, 2, 4, 2, 3]
assert answer_final_prices([10, 1, 1, 6]) == [9, 0, 1, 6]
assert answer_previous_smaller([3, 7, 8, 4]) == [-1, 3, 7, 3]
assert answer_previous_smaller([1, 1, 1]) == [-1, -1, -1]
assert answer_daily_temperatures([73,74,75,71,69,72,76,73]) == [1,1,4,2,1,1,0,0]
assert answer_daily_temperatures([30, 60, 90]) == [1, 1, 0]
assert answer_circular_final_prices([4, 6, 2, 3]) == [2, 4, 2, 1]
assert answer_circular_final_prices([5, 5]) == [0, 0]
assert answer_circular_final_prices([7]) == [7]

# Đối chiếu bài 6 với brute force vòng tròn
def brute_circular(prices):
    n = len(prices); res = []
    for i in range(n):
        val = prices[i]
        for step in range(1, n):
            j = (i + step) % n
            if prices[j] <= prices[i]:
                val -= prices[j]; break
        res.append(val)
    return res
random.seed(7)
for _ in range(1000):
    arr = [random.randint(1, 5) for _ in range(random.randint(1, 10))]
    assert answer_circular_final_prices(arr) == brute_circular(arr), arr
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

**Template monotonic stack "phần tử ≤ gần nhất bên phải":**
```python
class Solution:
    def finalPrices(self, prices: list[int]) -> list[int]:
        answer = prices[:]
        stack = []                          # chỉ số đang chờ, giá tăng dần
        for j, p in enumerate(prices):
            while stack and prices[stack[-1]] >= p:
                answer[stack.pop()] -= p    # j là đáp án của đỉnh
            stack.append(j)
        return answer
```

**Cách tư duy 4 bước:** (1) ai đang "chờ" đáp án? → cho vào stack; (2) phần tử mới trả lời cho ai? → điều kiện pop; (3) còn sót trong stack → giá trị mặc định; (4) đếm push/pop để thấy `O(n)`.

**Checklist trước khi submit:**
- [ ] Dùng `>=` (giá **bằng** vẫn được giảm)?
- [ ] Lấy món **gần nhất**, không phải rẻ nhất?
- [ ] Stack lưu **chỉ số** (để ghi vào `answer[i]`)?
- [ ] `stack and ...` đặt trước để tránh `IndexError`?
- [ ] Tạo bản sao `prices[:]`, không sửa input?
- [ ] Món còn trong stack giữ nguyên giá?